# Подготовка модели распознавания рукописных букв и цифр

In [1]:
# Импортируем необходимые библиотеки
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import EMNIST
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from torchvision.datasets import EMNIST


In [3]:
# Загружаем датасет
dataset = EMNIST(root='data/', split='balanced', download=False, train=True, transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(10),  # поворот на случайный угол от -10 до 10 градусов
]))
train_loader = DataLoader(dataset, batch_size=1024, shuffle=True)

dataset = EMNIST(root='data/', split='balanced', download=False, train=False, transform=transforms.ToTensor())
val_loader = DataLoader(dataset, batch_size=1024, shuffle=False)

In [4]:
# Определяем модель
class CNN(nn.Module):
    def __init__(self, n_classes):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)
        self.fc1 = nn.Linear(7 * 7 * 64, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, out_features=n_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        x = x.view(-1, 7 * 7 * 64)
        x = self.fc1(x)
        x = self.relu3(x)
        x = self.fc2(x)
        return x

In [5]:
# Функция для вычисления точности
def calculate_accuracy(model, data_loader):
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return correct / total

In [6]:
def train_model(model, train_loader, val_loader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for i, data in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")):
            inputs, labels = data
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        print(f"Epoch {epoch+1} loss: {running_loss/len(train_loader)}")

        # Вычисляем точность на валидационной выборке
        model.eval()
        val_accuracy = calculate_accuracy(model, val_loader)
        print(f"Validation accuracy: {val_accuracy}")

In [7]:
# Инициализируем модель и оптимизатор
model = CNN(47)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

n_epoch = 50

train_model(model, train_loader, val_loader, optimizer, criterion, n_epoch)

Epoch 1/50: 100%|██████████| 111/111 [01:50<00:00,  1.00it/s]


Epoch 1 loss: 1.6740153979610752
Validation accuracy: 0.7255851063829787


Epoch 2/50: 100%|██████████| 111/111 [01:51<00:00,  1.01s/it]


Epoch 2 loss: 0.745779165276536
Validation accuracy: 0.8068085106382978


Epoch 3/50: 100%|██████████| 111/111 [02:12<00:00,  1.19s/it]


...

Epoch 48 loss: 0.23046379932412156
Validation accuracy: 0.8767553191489361


Epoch 49/50: 100%|██████████| 111/111 [01:52<00:00,  1.01s/it]


Epoch 49 loss: 0.22604458987175882
Validation accuracy: 0.8776063829787234


Epoch 50/50: 100%|██████████| 111/111 [01:52<00:00,  1.01s/it]


Epoch 50 loss: 0.22789126043921118
Validation accuracy: 0.8776595744680851


In [10]:
# Сохраняем обученную модель
torch.save(model.state_dict(), 'myapp\model.ckpt')
